# BusPulse: first look at 30 October 2026
Bronze Parquet tables built from the raw Entur data in Swift.
Sampled: one estimated-timetable snapshot every 10 minutes,
vehicle positions every 5 minutes.

## 1. Check the environment

In [7]:
from pathlib import Path
import pandas as pd

DATA = Path(r"C:\Users\syeds\Documents\Project\Entur-Buspulse\tables\parquet_2026-10-30")

stop_visits = pd.read_parquet(DATA / "stop_visits.parquet")
vehicles = pd.read_parquet(DATA / "vehicle_positions.parquet")
alerts = pd.read_parquet(DATA / "service_alerts.parquet")
routes = pd.read_parquet(DATA / "gtfs_routes.parquet")
stops = pd.read_parquet(DATA / "gtfs_stops.parquet")
trips = pd.read_parquet(DATA / "gtfs_trips.parquet")

for name, df in [("stop_visits", stop_visits), ("vehicles", vehicles), ("alerts", alerts),
                 ("routes", routes), ("stops", stops), ("trips", trips)]:
    print(f"{name:12} {df.shape[0]:>10,} rows")

stop_visits   1,688,126 rows
vehicles         53,443 rows
alerts               75 rows
routes              318 rows
stops             7,673 rows
trips            13,116 rows


## 2. Load the tables
Loads the bronze tables and prints their sizes, to check they match the build log.

In [8]:
import pandas as pd

stop_visits = pd.read_parquet("stop_visits.parquet")
vehicles = pd.read_parquet("vehicle_positions.parquet")
alerts = pd.read_parquet("service_alerts.parquet")
routes = pd.read_parquet("gtfs_routes.parquet")
stops = pd.read_parquet("gtfs_stops.parquet")

for name, df in [("stop_visits", stop_visits), ("vehicles", vehicles), ("alerts", alerts),
                 ("routes", routes), ("stops", stops)]:
    print(f"{name:12} {df.shape[0]:>10,} rows  {df.shape[1]} columns")

stop_visits   1,688,126 rows  26 columns
vehicles         53,443 rows  13 columns
alerts               75 rows  17 columns
routes              318 rows  7 columns
stops             7,673 rows  12 columns


## 3. What's in `stop_visits`
One row per stop visit per snapshot. `recorded` = the bus has passed (actual times),
`estimated` = still to come (predicted times). Also shows how complete each column is.

In [9]:
print(stop_visits["call_type"].value_counts(), "\n")
print(stop_visits["vehicle_mode"].value_counts(), "\n")
print("lines:", stop_visits["line_ref"].nunique())
print("journeys:", stop_visits["journey_ref"].nunique())
print("snapshots:", stop_visits["snapshot_time"].nunique(),
      stop_visits["snapshot_time"].min(), "->", stop_visits["snapshot_time"].max())
stop_visits.isna().mean().round(2).sort_values()      # share of empty values per column

call_type
estimated    1218037
recorded      470089
Name: count, dtype: int64 

vehicle_mode
bus      1631873
ferry      56006
Name: count, dtype: int64 

lines: 266
journeys: 5708
snapshots: 143 2026-10-30 00:00:39+00:00 -> 2026-10-30 23:55:19+00:00


snapshot_time            0.00
recorded_at              0.00
journey_ref              0.00
data_frame_ref           0.00
line_ref                 0.00
direction_ref            0.00
vehicle_mode             0.00
operator_ref             0.00
monitored                0.00
call_type                0.00
stop_order               0.00
stop_ref                 0.00
source_file              0.00
stop_name                0.03
aimed_departure          0.04
aimed_arrival            0.04
arrival_status           0.31
destination              0.31
expected_arrival         0.31
expected_departure       0.32
departure_status         0.34
actual_departure         0.73
actual_arrival           0.74
journey_cancelled        0.97
call_cancelled           0.97
prediction_inaccurate    0.97
dtype: float64

## 4. Final state of each stop visit
The same journey appears in every snapshot. Keep only the latest version
of each stop visit (journey + stop order), which holds the actual times.

In [10]:
latest = (stop_visits
          .sort_values("snapshot_time")
          .drop_duplicates(subset=["data_frame_ref", "journey_ref", "stop_order"], keep="last"))

print(f"{len(stop_visits):,} rows -> {len(latest):,} unique stop visits")
print(latest["call_type"].value_counts())

1,688,126 rows -> 157,480 unique stop visits
call_type
recorded     154039
estimated      3441
Name: count, dtype: int64


## 5. Delay per line
Delay = actual time − planned (aimed) time, in seconds, using departure when
available, otherwise arrival. Only buses with actual times. Line numbers come
from the GTFS routes table. Lines with fewer than 50 stop visits are hidden.

In [11]:
def ts(col):
    return pd.to_datetime(latest[col], utc=True, format="ISO8601")

actual = ts("actual_departure").fillna(ts("actual_arrival"))
aimed = ts("aimed_departure").fillna(ts("aimed_arrival"))
latest["delay_s"] = (actual - aimed).dt.total_seconds()

# real line numbers from GTFS
latest = latest.merge(routes[["route_id", "route_short_name"]],
                      left_on="line_ref", right_on="route_id", how="left")

buses = latest[(latest["call_type"] == "recorded") & (latest["vehicle_mode"] == "bus")
               & latest["delay_s"].notna()]
print(f"{len(buses):,} actual bus stop visits")
print(buses["delay_s"].describe().round(0))

(buses.groupby("route_short_name")["delay_s"]
      .agg(visits="count", mean_delay="mean", median_delay="median")
      .query("visits >= 50")
      .sort_values("mean_delay", ascending=False)
      .round(0)
      .head(15))

153,180 actual bus stop visits
count    153180.0
mean        194.0
std         346.0
min       -2927.0
25%          40.0
50%         129.0
75%         273.0
max       10217.0
Name: delay_s, dtype: float64


,visits,mean_delay,median_delay
route_short_name,,,
246,504,842.0,199.0
SK0302,79,539.0,512.0
SK4910,351,406.0,304.0
81,81,395.0,229.0
58,200,383.0,235.0
SK4924,59,366.0,412.0
X39,469,365.0,336.0
SK0201,108,359.0,369.0
42,6291,351.0,252.0


## 6.Link between real time feeds and GTFS

In [14]:
from pathlib import Path
import pandas as pd

DATA = Path(".")

stop_visits = pd.read_parquet(DATA / "stop_visits.parquet")
vehicles = pd.read_parquet(DATA / "vehicle_positions.parquet")
routes = pd.read_parquet(DATA / "gtfs_routes.parquet")
stops = pd.read_parquet(DATA / "gtfs_stops.parquet")
trips = pd.read_parquet(DATA / "gtfs_trips.parquet")


def coverage(values, known):
    v = set(values.dropna())
    return f"{len(v & set(known))}/{len(v)} ({len(v & set(known)) / len(v):.0%})"


print("line_ref    -> routes.route_id:", coverage(stop_visits["line_ref"], routes["route_id"]))
print("journey_ref -> trips.trip_id:  ", coverage(stop_visits["journey_ref"], trips["trip_id"]))
print("stop_ref    -> stops.stop_id:  ", coverage(stop_visits["stop_ref"], stops["stop_id"]))
print("vehicles    -> routes.route_id:", coverage(vehicles["line_ref"], routes["route_id"]))

line_ref    -> routes.route_id: 266/266 (100%)
journey_ref -> trips.trip_id:   5708/5708 (100%)
stop_ref    -> stops.stop_id:   4508/4508 (100%)
vehicles    -> routes.route_id: 264/265 (100%)
